# Agent Memory: 개념과 동작 구조

본 노트북은 에이전트 메모리의 구성 요소와 정보 처리 과정을 정의한다. 분석 대상은 `neo4j-agent-memory`의 Python SDK 0.8.0과 Neo4j에 직접 연결하는 Bolt 백엔드이다. 계층의 개념적 구분과 현재 구현의 동작 범위를 구별하여 기술한다.

주요 논점은 기억 계층, 그래프 표현, 저장과 추출의 결합, 검색 범위, 데이터 삭제, 클라이언트 생명주기이다. 구체적인 실험은 [단기 기억](01_short_term_memory.ipynb), [엔티티와 관계](02_long_term_memory_entity.ipynb), [Fact](03_long_term_memory_fact.ipynb) 노트북에서 각각 수행한다.

## 에이전트 메모리의 목적

에이전트 메모리는 이전 상호작용과 실행 결과를 외부 저장소에 기록하고, 이후 작업에 필요한 정보를 검색하여 제공하는 구성 요소이다. 본 실험에서는 Neo4j가 저장소 역할을 수행한다.

정보 처리는 다음 세 연산으로 구분한다.

| 연산 | 목적 | 산출물 |
|---|---|---|
| 기록 | 메시지와 구조화된 정보를 보존한다. | DB의 노드, 관계, 속성 |
| 검색 | 현재 질의와 관련된 정보를 선택한다. | 메시지, 엔티티, 선호도, 작업 기록 등의 검색 결과 |
| 컨텍스트 구성 | 검색 결과를 후속 모델 호출에 전달할 형태로 조합한다. | 프롬프트에 삽입할 문자열 |

DB에 정보를 저장하는 연산과 LLM의 모델 가중치를 갱신하는 연산은 별개이다. 저장된 정보가 응답에 반영되려면 검색 결과를 후속 모델 호출에 전달하는 애플리케이션 로직이 필요하다. `get_context()`의 반환값은 이 과정에 사용되는 컨텍스트 문자열이다. [프로젝트 개요](https://neo4j.com/labs/agent-memory/)

## 기억 계층의 구분

`MemoryClient`는 동일한 그래프에 저장된 정보를 세 인터페이스로 구분하여 제공한다.

| 계층 | 주요 정보 | 대표 객체 | 주요 용도 |
|---|---|---|---|
| 단기 기억 (`short_term`) | 대화 원문, 발화 역할, 순서, 메타데이터 | `Conversation`, `Message` | 대화 이력 조회와 관련 메시지 검색 |
| 장기 기억 (`long_term`) | 엔티티와 관계, 선호도, 사실 | `Entity`, `Preference`, `Fact` | 재사용 가능한 구조화된 지식의 관리와 검색 |
| 추론 기억 (`reasoning`) | 명시적으로 기록한 작업 단계, 도구 호출, 실행 결과 | `ReasoningTrace`, `ReasoningStep`, `ToolCall` | 작업 경로와 도구 사용 결과의 분석 |

여기서 단기·장기의 구분은 정보의 역할과 표현 방식에 관한 분류이다. 단기 기억도 DB에 영속 저장되며, 커널 종료만으로 삭제되지 않는다. 이름만으로 자동 만료 정책이 정의되는 것은 아니다.

추론 기억은 애플리케이션이 기록한 실행 이력이다. 일반 메시지 저장만으로 모든 작업 기록이 자동 생성되거나 모델 내부의 비공개 추론이 추출되는 것은 아니다. 이 분류는 인간 기억 체계와 일대일로 대응하는 이론적 분류가 아니라 라이브러리의 기능적 구분이다. [기억 계층 정의](https://neo4j.com/labs/agent-memory/explanation/memory-types/)

## 그래프 기반 표현

대화와 추출 지식은 서로 연결된 그래프 요소로 표현된다. 다음은 단기 기억과 엔티티 사이의 핵심 관계를 나타낸 개념도이다.

```text
(Conversation)
      │ HAS_MESSAGE
      ▼
  (Message) ──MENTIONS──▶ (Entity)
                             │ RELATED_TO {type: ...}
                             ▼
                          (Entity)
```

`Conversation`은 메시지를 묶는 단위이고, `Message`는 원문과 발화 정보를 보존한다. `MENTIONS`는 메시지에서 언급된 엔티티와 원문을 연결한다. 엔티티 사이의 관계는 `RELATED_TO` 간선과 관계 유형 등의 속성으로 표현된다. [그래프 구조 개요](https://neo4j.com/labs/agent-memory/)

예를 들어 GCP와 Cloud DNS를 언급한 문장은 메시지로 보존되며, 추출 결과에 따라 두 개념의 엔티티와 관계가 저장될 수 있다. 구체적인 유형과 관계는 추출 결과에 의존하므로 항상 동일한 그래프가 생성되는 것은 아니다.

원문과 추출 지식은 서로 다른 수명과 검증 기준을 가진다. 특히 엔티티 유형이나 관계가 잘못 추출될 수 있으므로, 그래프에 저장되었다는 사실만으로 해당 지식의 정확성이 보장되지는 않는다.

## 메시지 저장과 지식 추출

### 기본 저장 경로

현재 실험의 `short_term.add_message()`는 다음 순서로 동작한다.

1. 세션에 대응하는 대화를 조회하거나 생성한다.
2. 메시지 임베딩을 생성하고 원문 메시지를 저장한다.
3. 추출이 활성화된 경우 설정된 추출 파이프라인을 실행한다.
4. 추출된 엔티티를 생성하거나 기존 엔티티와 연결하고, 메시지와의 `MENTIONS` 관계를 저장한다.
5. 유효한 엔티티 간 관계가 존재하면 해당 관계를 저장한다.

따라서 `short_term`을 통한 입력은 장기 기억의 재료를 생성할 수 있다. 대화 원문은 유지되며, 추출된 지식이 별도의 그래프 요소로 추가되는 구조이다. 시간 경과에 따른 자동 승격으로 해석해서는 안 된다. [메시지 저장 API](https://neo4j.com/labs/agent-memory/reference/api/short-term-messages/)

현재 프로젝트는 GLiNER2.5와 LLM을 사용하는 추출 파이프라인을 구성한다. 이 단계는 문자열 저장에 비해 추가적인 계산과 외부 호출을 요구한다. GLiNER2.5는 엔티티와 관계를 추출하며, LLM 단계는 설정에 따라 선호도 후보도 추출할 수 있다. [추출 파이프라인](https://neo4j.com/labs/agent-memory/explanation/extraction-pipeline/)

다만 설치된 0.8.0의 `add_message()` 저장 경로는 추출 결과 중 엔티티와 관계를 영속화한다. 선호도 후보가 추출되었다는 사실과 `Preference` 노드가 저장되었다는 사실은 구별해야 한다. `Preference`와 `Fact`의 명시적 관리는 각각 장기 기억 API의 책임이다.

### 추출 방식의 제어

| 설정 | 저장 시 동작 |
|---|---|
| `extraction_mode="auto"` | `extract_entities=True`이고 추출기가 구성된 경우 자동 추출을 실행한다. 기본값이다. |
| `extraction_mode="skip"` | 추출을 생략하며 해당 호출에서 `MENTIONS` 관계를 생성하지 않는다. |
| `extraction_mode="explicit"` | 자동 추출 대신 `explicit_mentions`로 전달한 엔티티 참조를 처리한다. |
| `generate_embedding=True` | 메시지의 임베딩을 생성하며, 추출한 엔티티를 저장할 때도 임베딩을 생성한다. |

`skip`은 추출 단계만 생략한다. 메시지 저장과 메시지 임베딩 생성은 별도 제어 대상이다. 이미 저장된 엔티티나 관계를 제거하는 효과는 없다. [옵션 정의](https://neo4j.com/labs/agent-memory/reference/api/short-term-messages/)

```python
# 대화 원문과 검색용 임베딩을 저장하고 자동 추출은 생략한다.
await memory_client.short_term.add_message(
    session_id=session_id,
    role="user",
    content=message,
    extraction_mode="skip",
    generate_embedding=True,
)
```

## 이력 조회와 컨텍스트 검색

이력 조회는 특정 대화의 메시지를 읽는 연산이다. 의미 기반 검색은 질의 임베딩과 저장된 임베딩의 유사도를 사용하여 관련 항목을 선택하는 연산이다. 두 연산은 조회 기준과 범위가 다르다.

| 연산 | 선택 기준 | 현재 Bolt 구현의 범위 |
|---|---|---|
| `get_conversation(session_id)` | 세션 식별자 | 지정한 세션의 대화 |
| `short_term.search_messages(query)` | 메시지 임베딩 유사도 | 전체 메시지 검색; 인자로 받은 `session_id`도 현재 구현에서 적용되지 않음 |
| `get_context(query, session_id=...)` | 최근 이력과 각 기억 계층의 관련 정보 | 세션 이력과 세션 간 검색 결과의 조합 |

현재 `get_context()`의 출력은 다음 요소로 구성된다.

| 출력 구역 | 의미 |
|---|---|
| `Recent Conversation` | 지정 세션의 최근 메시지 |
| `Relevant Past Messages` | 전체 세션에서 질의와 관련 있는 메시지 |
| `Relevant Entities` | 장기 기억에 저장된 관련 엔티티 |
| `User Preferences` | 장기 기억에 저장된 관련 선호도 |

결과가 존재하는 구역만 출력된다. 유사도는 검색상 관련성의 척도이며, 문장의 사실성이나 추출 결과의 정확성에 대한 확률로 해석하지 않는다. [검색과 컨텍스트 구성](https://neo4j.com/labs/agent-memory/reference/api/short-term-search/)

### 저장 옵션과 조회 옵션의 독립성

`extraction_mode`는 기록 시점의 처리를 제어한다. `include_short_term`, `include_long_term`, `include_reasoning`은 조회 결과에 포함할 계층을 제어한다. 두 종류의 옵션은 서로 다른 시점에 적용된다.

| 저장 시 추출 | 조회 시 `include_long_term` | 해석 |
|---|---|---|
| `auto` | `True` | 추출 지식이 저장될 수 있으며, 관련 장기 기억이 조회 결과에 포함된다. |
| `auto` | `False` | 추출 지식은 저장될 수 있으나 해당 컨텍스트 조회에서는 제외된다. |
| `skip` | `True` | 현재 메시지에서 새 지식을 추출하지 않지만, 기존 장기 기억은 검색될 수 있다. |
| `skip` | `False` | 현재 메시지의 추출을 생략하고 해당 조회에서 장기 기억을 제외한다. |

`include_long_term=False`는 저장된 엔티티를 삭제하지 않는다. `include_short_term=True`도 현재 세션만 검색하도록 보장하지 않는다. [컨텍스트 API](https://neo4j.com/labs/agent-memory/reference/api/memory-client/)

## 세션과 검색 범위

`session_id`는 대화의 논리적 구분을 위한 식별자이다. 본 실험의 `01_memory_basics`는 하나의 대화를 식별하지만, DB 전체의 메모리 범위와 동일하지 않다.

현재 Bolt 구현에서 세션별 이력 조회와 삭제는 지정한 세션을 대상으로 수행된다. 반면 의미 기반 메시지 검색과 장기 기억 검색은 다른 세션에서 생성된 정보를 포함할 수 있다. 세션 식별자만으로 데이터 접근 경계가 정의되지는 않는다. [검색 범위](https://neo4j.com/labs/agent-memory/reference/api/short-term-search/)

예를 들어 `01_memory_basics`를 삭제한 뒤에도 이전 `lab-session`에 동일한 문장이 남아 있으면 `Relevant Past Messages`에 나타날 수 있다. 이는 세션 삭제와 전체 검색이 서로 다른 범위를 사용하기 때문이다.

또한 동일한 문장을 `add_message()`로 반복 저장하면 별도의 메시지가 생성된다. 세션별 메시지 수, 중복 기록, 공유 엔티티의 존재 여부를 구분하여 실험 결과를 해석해야 한다.

## 삭제와 연결 종료

| 연산 | 적용 대상 | 적용 후 남는 정보 |
|---|---|---|
| `clear_session(session_id)` | 지정 세션의 대화와 연결된 메시지 | 다른 세션의 메시지, 공유 엔티티, 별도로 저장된 선호도·사실 등 |
| `close()` | 클라이언트의 연결과 백그라운드 작업 | DB에 영속 저장된 메모리 |
| 커널 재시작 | 노트북 프로세스의 변수와 실행 상태 | DB에 영속 저장된 메모리 |

`clear_session()`은 메시지와 엔티티 사이의 연결도 제거하지만, 언급된 엔티티 자체는 유지한다. 별도로 기록된 추론 이력 전체를 삭제하는 API도 아니므로 완전한 DB 초기화와 구별한다. [세션 삭제 API](https://neo4j.com/labs/agent-memory/reference/api/short-term-sessions/)

삭제 후 `get_conversation()`으로 메시지 수가 0임을 확인하는 것은 해당 세션의 이력이 비어 있음을 검증한다. 전체 검색 결과가 비어 있거나 모든 기억 계층이 초기화되었음을 검증하는 것은 아니다.

## 클라이언트 생명주기와 실행 비용

클라이언트의 생명주기는 설정 구성, 객체 생성, 연결, 작업 수행, 종료로 구분한다.

```text
MemorySettings → MemoryClient → connect()
                                    │
                            저장 · 조회 · 검색
                                    │
                                 close()
```

Bolt 연결 시 스키마와 메모리 구성 요소가 초기화된다. 여러 셀에서 동일한 클라이언트를 재사용하면 연결과 초기화를 반복하는 비용을 줄일 수 있다. 현재 노트북에서는 준비 단계에서 연결하고, 정리 단계에서 명시적으로 종료한다. [연결과 종료 API](https://neo4j.com/labs/agent-memory/reference/api/memory-client/)

실행 비용은 다음과 같이 구분한다.

| 구간 | 주요 비용 요소 |
|---|---|
| 연결·초기화 | 원격 DB 연결, 스키마 확인, 추출기 구성 및 모듈 로딩 |
| 메시지 저장 | 임베딩 API, GLiNER 추론, LLM 추출, 그래프 기록 |
| 컨텍스트 검색 | 질의 임베딩, 벡터 검색, 결과 조회와 문자열 구성 |

현재 구현은 컨텍스트 검색 과정에서 메시지·선호도·엔티티 검색에 동일 질의를 각각 임베딩한다. 따라서 전체 실행 시간을 해석할 때 초기화 비용과 검색 비용을 분리해야 한다. 최초 저장 시 모델 로딩 비용과 이후 추론 비용 역시 구별할 필요가 있다.

## 후속 실험의 관찰 항목

[01. Short-Term Memory](01_short_term_memory.ipynb)는 메시지 저장, 세션 이력 조회, 컨텍스트 검색, 세션 삭제, 연결 종료를 순서대로 수행한다. 실험 결과는 다음 기준으로 해석한다.

| 조건 또는 조작 | 관찰 대상 | 해석 기준 |
|---|---|---|
| 빈 세션에 메시지 4개를 한 번 저장 | 해당 세션의 반환 메시지 수 | 조회 제한 내에서 4개가 반환된다. |
| 동일 저장 셀 재실행 | 메시지 수와 검색 중복 | 별도 메시지가 추가될 수 있다. |
| `extraction_mode="skip"`으로 저장 | 현재 메시지의 추출 경로 | 새 자동 추출과 `MENTIONS` 생성이 생략된다. |
| `include_long_term=False`로 조회 | 컨텍스트 출력 구역 | 장기 기억 구역이 제외되며 저장 데이터는 유지된다. |
| 현재 세션 삭제 | 해당 세션의 메시지 수 | 0개가 반환되어도 다른 세션과 장기 기억은 남을 수 있다. |
| 클라이언트 종료 | 연결 상태 | 연결 자원은 정리되며 저장 데이터는 유지된다. |

추출 유무를 비교할 때는 기존 엔티티가 이미 저장되어 있는지 확인해야 한다. 또한 벡터 검색 결과와 엔티티 분류는 입력 문장, 모델, 설정 및 기존 그래프 상태에 영향을 받으므로 고정된 정답으로 간주하지 않는다.

## 참고 문헌 및 구현 근거

본 문서의 API 동작은 프로젝트에 설치된 `neo4j-agent-memory` 0.8.0의 소스와 아래 공식 문서를 기준으로 기술한다. 확인일은 2026-11-06이다.

1. Neo4j, [Understanding the three memory types](https://neo4j.com/labs/agent-memory/explanation/memory-types/).
2. Neo4j, [How entity extraction works](https://neo4j.com/labs/agent-memory/explanation/extraction-pipeline/).
3. Neo4j, [Conversation and message operations](https://neo4j.com/labs/agent-memory/reference/api/short-term-messages/).
4. Neo4j, [Message search and context](https://neo4j.com/labs/agent-memory/reference/api/short-term-search/).
5. Neo4j, [Sessions, summaries, and maintenance](https://neo4j.com/labs/agent-memory/reference/api/short-term-sessions/).
6. Neo4j, [MemoryClient API reference](https://neo4j.com/labs/agent-memory/reference/api/memory-client/).

구현 확인 대상은 `MemoryClient.connect/get_context/close`, `ShortTermMemory.add_message/_extract_and_link_entities/get_context/clear_session`, `LongTermMemory.get_context`이다. 버전 변경 시 검색 범위와 자동 저장 범위를 다시 확인해야 한다.